# AI Movie Studio — LOCAL AI / NO API KEYS
All inference runs inside this Colab runtime using open-weight models. Public model weights are downloaded once from Hugging Face and cached on Google Drive; generation itself does not call OpenAI, Replicate, ElevenLabs or any hosted inference API.

**Recommended:** GPU runtime. First launch downloads several GB of model weights. Colab Free may disconnect or run out of VRAM; use shorter clips and generate shots one at a time.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/ai-movie-studio')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PandaCwel/Ai-video-maker.git',str(repo)],check=True)
os.chdir(repo)
print('Repository:', repo)


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -r requirements.txt
!pip -q install -r requirements-local.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
data_root = Path('/content/drive/MyDrive/AI-Movie-Studio')
(data_root/'data').mkdir(parents=True,exist_ok=True)
(data_root/'models').mkdir(parents=True,exist_ok=True)
os.environ['STUDIO_DATA_DIR'] = str(data_root/'data')
os.environ['HF_HOME'] = str(data_root/'models'/'huggingface')
os.environ['TORCH_HOME'] = str(data_root/'models'/'torch')
os.environ.setdefault('LOCAL_LLM_MODEL','Qwen/Qwen2.5-3B-Instruct')
os.environ.setdefault('LOCAL_IMAGE_MODEL','stable-diffusion-v1-5/stable-diffusion-v1-5')
os.environ.setdefault('LOCAL_VIDEO_MODEL','guoyww/animatediff-motion-adapter-v1-5-2')
os.environ.setdefault('LOCAL_MUSIC_MODEL','facebook/musicgen-small')
print('Projects and downloaded model cache will persist in Google Drive. No API keys required.')


In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))
else:
    print('WARNING: CPU inference is possible for some models but video generation will be extremely slow.')


## Local lip-sync (optional)
The app supports local Wav2Lip inference. The Wav2Lip checkpoint is not redistributed by this repository. Obtain it lawfully, put it in Drive, then set `WAV2LIP_CHECKPOINT` to its path and `WAV2LIP_REPO` to the local `inference.py`. This feature does not use an API.

In [ ]:
# Optional: configure your locally stored Wav2Lip checkpoint.
wav2lip_checkpoint = data_root/'models'/'wav2lip_gan.pth'
if wav2lip_checkpoint.exists():
    os.environ['WAV2LIP_CHECKPOINT'] = str(wav2lip_checkpoint)
    print('Wav2Lip checkpoint found.')
else:
    print('Wav2Lip checkpoint not installed; other local engines remain available.')


In [ ]:
import subprocess, time
server = subprocess.Popen([sys.executable,'-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'])
time.sleep(5)
from google.colab.output import eval_js
studio_url = eval_js('google.colab.kernel.proxyPort(8000)')
print('AI Movie Studio:',studio_url)
print('API docs:',studio_url+'/docs')


## Local engines and practical limits
- **Screenplay / AI Director:** Qwen 2.5 3B Instruct, local Transformers inference (4-bit quantization when available).
- **Images:** Stable Diffusion 1.5 through Diffusers.
- **Video:** AnimateDiff motion adapter + Stable Diffusion 1.5. It generates short animated clips, not a full 30-minute feature in one pass.
- **Voice cloning / speech:** Coqui XTTS-v2 with an uploaded voice sample. Only use voices you own or have permission to clone.
- **Music:** MusicGen-small.
- **Lip-sync:** local Wav2Lip, requires a separately supplied checkpoint.
- **Editing:** FFmpeg local concat/export.

No paid inference API and no API keys. Model weights are downloaded from public repositories and then run locally. Model licenses still apply. Colab Free GPU availability, VRAM, session limits and Drive storage are not guaranteed. A 25–30 minute episode requires many separate shots, long generation time and sufficient compute; it is not guaranteed on a free runtime.